# Augplot: start with auto, then add the hard part in one sentence

Two examples where a short refinement replaces substantial plotting code: explore penguin measurements, then reshape monthly flight totals into an annotated heatmap. Both are real sample datasets loaded through Augplot from Seaborn's public catalog.

**Setup:** install using the [README](../README.md#install), select your Augplot kernel, and run the first cell. It selects `openai/gpt-5.6-terra` and asks for your API key with hidden input.

First run: four generations, plus one if you enable Plotly; each may need one repair request. Identical reruns use saved code. Loading a dataset needs internet access the first time; Seaborn caches it locally. Your provider receives a data profile, and generated Python runs locally without a security sandbox.

In [ ]:
import os
from getpass import getpass

os.environ["AUGPLOT_MODEL"] = "openai/gpt-5.6-terra"
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

In [ ]:
import augplot as ap

## 1. Start from a penguin measurement comparison

The Palmer penguins dataset has body and bill measurements for three species across three islands. Load it without a separate Seaborn import, then ask Augplot for a useful first view.

The next cell adds facets, trends, and another visual encoding while keeping the prompt short.

In [ ]:
penguins = ap.load_sns_dataset("penguins")
plt = ap.plot(
    penguins,
    backend="seaborn",
    prompt="Compare bill length and depth across penguin species.",
)

In [ ]:
print(plt.explanation)
print("Reused saved code:", plt.cache_hit)

## 2. Add the hard part in one sentence

This refinement coordinates small multiples, per-species trend lines, consistent axes, and marker shapes for sex—details that otherwise require several plotting calls and legend cleanup.

In [ ]:
plt.refine(
    "Use one panel per species with shared axes, add a linear trend in each panel, "
    "and keep sex visible with marker shape."
)

In [ ]:
# The generated function and its saved version are available for inspection.
print("Saved source:", plt.history_path)
# print(plt.code)

## 3. A real subset, same function

Use only penguins observed on Biscoe Island to show that `render()` can apply the generated function to compatible data without another model call. Export the function under a readable name when you're happy with it.

In [ ]:
biscoe_penguins = penguins[penguins["island"] == "Biscoe"]
plt.render(biscoe_penguins, title="Penguin bills on Biscoe Island")

In [ ]:
from pathlib import Path

# Choose a fresh filename on reruns so we do not overwrite existing code.
export_path = Path("vis_utils.py")
version = 2
while export_path.exists():
    export_path = Path(f"vis_utils_{version}.py")
    version += 1

plt.save(export_path, function_name="plot_penguin_bills")

In [ ]:
# The printed snippet works for normal imports. Here we load the chosen file
# directly so this cell also works when a rerun selected a numbered filename.
import importlib.util

spec = importlib.util.spec_from_file_location(export_path.stem, export_path)
vis_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vis_utils)

fig = vis_utils.plot_penguin_bills(biscoe_penguins, figsize=(11, 5))
fig

## 4. Start from monthly airline passengers

The flights dataset contains monthly passenger totals from 1949 through 1960. Start with a line chart so the overall growth and seasonal cycles are visible.

Because Seaborn assigns an ordered categorical type to `month`, the calendar order is available in the data profile.

In [ ]:
flights = ap.load_sns_dataset("flights")
flight_viz = ap.plot(
    flights,
    backend="seaborn",
    prompt="Plot monthly airline passengers over time, with one line per year.",
)

## 5. Turn it into a seasonal heatmap in one sentence

A heatmap makes both long-term growth and recurring seasonality easier to scan. The refinement must pivot the data, preserve calendar order, annotate cells, and identify the peak.

In [ ]:
flight_viz.refine(
    "Turn this into a year-by-month heatmap, preserve calendar month order, "
    "annotate each cell, and emphasize the busiest month."
)

## 6. Optional: make the heatmap interactive

Set the toggle to `True` to generate the same real dataset with Plotly hover details.

In [ ]:
RUN_PLOTLY = False

if RUN_PLOTLY:
    interactive_viz = ap.plot(
        flights,
        prompt="Show a year-by-month passenger heatmap with hover details.",
        backend="plotly",
    )

**Rerunning:** run from the original `ap.plot()` cell to replay the same refinement. Repeating only `refine()` edits the current version again. Keep `.augplot/` with this notebook. [How history works](../docs/visualization-history.md).